# AI-RAG · Week 16 — Notebook 2 (Day 2)
## Port the Agent + Deterministic Guardrails + Resilience + Parallelism

Day 2: port the raw agent to a full LangGraph, add **two deterministic guardrail nodes** (PII check, cost limit), wire LangGraph's **resilience primitives** (`RetryPolicy`, fallback, `interrupt`/HITL), and use **`Send`** for parallel fan-out — plus quick previews of persistence and streaming. Runs **fully offline**.

> New LangGraph code, executed. Prerequisites re-establish the tools + mock from Notebook 1.

### Setup

In [1]:
# --- notebook setup -----------------------------------------------------------
# Installs LangGraph (pinned) + pydantic, writes the offline web fixtures, and builds
# the SQLite side-DB. LangGraph runs fully offline; the LLM "think" nodes use a mock,
# so there is no OpenAI/Tavily key and no cost.
import subprocess, sys
for pkg in ("langgraph>=1.0,<2", "pydantic>=2.5"):
    try:
        __import__(pkg.split(">")[0].split("=")[0].replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/web_fixtures.json").write_text(r"""{
  "ai agents": [
    {"title": "State of AI Agents 2026", "snippet": "Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator."},
    {"title": "Agent frameworks roundup", "snippet": "Raw loops and graph-based orchestration remain the two dominant patterns."}
  ],
  "openai api": [
    {"title": "OpenAI API pricing", "snippet": "gpt-4o-mini is positioned as the low-cost tier for high-volume workloads."}
  ],
  "cricket": [
    {"title": "Cricket World Cup — result", "snippet": "The most recent tournament concluded with a closely contested final."}
  ],
  "python libraries": [
    {"title": "Trending Python libraries", "snippet": "Data, LLM-tooling, and async libraries dominate this year's trending lists."}
  ],
  "announce": [
    {"title": "OpenAI announcements", "snippet": "Recent announcements focus on tool use, structured outputs, and cost reductions."}
  ],
  "_default": [
    {"title": "Web result", "snippet": "A relevant public-web snippet would appear here (offline fixture)."}
  ]
}""" + "\n")
import langgraph
print("langgraph", getattr(langgraph, "__version__", "?"), "· offline · fixtures written")


langgraph ? · offline · fixtures written


### Prerequisites — carried tool + LLM pieces

The tool interface, the four tools, and the mock LLM — carried from W13–W15. This week we rebuild the **agent loop** as a LangGraph.

In [2]:
"""src/agents/tools/base.py — the Tool contract.

A tool an LLM can call reliably is four things done well (the W13 principles):
  1. a good NAME       — a verb, specific, snake_case
  2. a typed SCHEMA    — a Pydantic model: required fields, sensible defaults
  3. a DESCRIPTION     — written for the *model*, saying WHEN to use the tool
  4. IDEMPOTENCY        — declared, so the loop knows if a retry is safe

Every tool subclasses `Tool`, sets those four, and implements `run`. The same
definition exports to both OpenAI function-calling and Anthropic tool-use schemas.
"""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel


class Tool:
    name: str = ""
    description: str = ""
    Args: type[BaseModel] = BaseModel
    idempotent: bool = True          # is calling this twice with the same args safe?
    has_side_effects: bool = False   # does it change external state?

    def run(self, **kwargs: Any) -> str:
        raise NotImplementedError

    # ── validation + execution ────────────────────────────────────────────
    def call(self, arguments: dict[str, Any]) -> str:
        """Validate arguments against the schema, then run."""
        args = self.Args(**(arguments or {}))
        return self.run(**args.model_dump())

    # ── provider schema exports ───────────────────────────────────────────
    def parameters_schema(self) -> dict:
        return self.Args.model_json_schema()

    def openai_schema(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name,
                             "description": self.description,
                             "parameters": self.parameters_schema()}}

    def anthropic_schema(self) -> dict:
        return {"name": self.name,
                "description": self.description,
                "input_schema": self.parameters_schema()}


In [3]:
"""src/agents/tools/calculate.py — a computation tool.

Deterministic, offline, and safe: it evaluates an arithmetic expression with a
small AST walker (no `eval`), so an LLM can hand it a calculation and get a
reliable number back. Idempotent and side-effect-free — the easy end of the
tool spectrum, and a good first example of a well-formed tool.
"""
from __future__ import annotations

import ast
import operator

from pydantic import BaseModel, Field

# Tool defined above

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos, ast.FloorDiv: operator.floordiv,
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression — only + - * / % ** ( ) and numbers")


class CalculateArgs(BaseModel):
    expression: str = Field(
        ..., description="An arithmetic expression, e.g. '1350 * 0.18' or '(42 + 8) / 2'.")


class CalculateTool(Tool):
    name = "calculate"
    description = ("Evaluate an arithmetic expression and return the numeric result. "
                   "Use this whenever a question needs exact math — sums, differences, "
                   "products, division, percentages, or powers. Do NOT use it to look up "
                   "facts; it only does arithmetic on numbers you provide.")
    Args = CalculateArgs
    idempotent = True
    has_side_effects = False

    def run(self, expression: str) -> str:
        try:
            result = _eval(ast.parse(expression, mode="eval"))
        except Exception as e:  # noqa: BLE001 - report cleanly to the agent
            return f"calculate error: {e}"
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"{expression} = {result}"


In [4]:
"""src/agents/tools/query_sql.py — SQL as a tool, safely.

Letting an LLM run SQL is powerful and dangerous. This tool ships the three rails
every SQL-as-a-tool needs in production:

  1. AUDIT LOGGING     — every query (allowed or denied) is written to audit_log
  2. READ-ONLY DEFAULT  — only a single SELECT / WITH…SELECT is permitted
  3. CONSTRAINED UPDATES — writes/DDL (INSERT/UPDATE/DELETE/DROP/…) are denied
                          outright (write access is a W18 topic, gated behind
                          injection defenses and a human-approval step)

Everything runs against a local SQLite database, so it works offline with no
credentials. The migration path to Postgres is a config change, not a rewrite
(see docs/runbooks/sql_as_a_tool_safety.md).
"""
from __future__ import annotations

import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, Field

# Tool defined above

DB_PATH = Path("data/capstone_side.db")

# Statements that change data or schema — denied by the read-only rail.
_FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "replace",
              "truncate", "attach", "detach", "pragma", "vacuum", "reindex",
              "grant", "revoke", "begin", "commit"}
_MAX_ROWS = 50


def _strip_comments(sql: str) -> str:
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    return sql.strip()


def is_read_only(sql: str) -> tuple[bool, str]:
    """Return (ok, reason). Only a single SELECT/WITH…SELECT is allowed."""
    clean = _strip_comments(sql)
    if not clean:
        return False, "empty query"
    statements = [s for s in clean.split(";") if s.strip()]
    if len(statements) > 1:
        return False, "multiple statements are not allowed"
    first = statements[0].lstrip().split(None, 1)[0].lower()
    if first not in ("select", "with"):
        return False, f"only SELECT queries are allowed (got '{first.upper()}')"
    words = set(re.findall(r"[a-z_]+", clean.lower()))
    hit = words & _FORBIDDEN
    if hit:
        return False, f"write/DDL keyword not permitted: {', '.join(sorted(hit))}"
    return True, "ok"


class QuerySqlArgs(BaseModel):
    sql: str = Field(..., description="A single read-only SQL SELECT query over the "
                                      "document_metadata table.")
    purpose: str = Field("", description="One phrase describing what the query is for "
                                         "(recorded in the audit log).")


class QuerySqlTool(Tool):
    name = "query_sql"
    description = (
        "Run a read-only SQL SELECT against the knowledge-base metadata database and "
        "return the rows. Use this for questions about the DOCUMENTS THEMSELVES — how "
        "many documents exist, which department owns them, their type, version, "
        "chunk_count, or when they were last updated. Table: document_metadata"
        "(id, source, title, doc_type, department, version, updated_at, chunk_count). "
        "Do NOT use it to answer questions about document CONTENTS — only their metadata.")
    Args = QuerySqlArgs
    idempotent = True            # SELECTs don't change state
    has_side_effects = False

    def __init__(self, db_path: Path | str = DB_PATH):
        self.db_path = Path(db_path)

    def _audit(self, request: str, sql: str, decision: str,
               row_count: int | None, error: str | None) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute(
            "INSERT INTO audit_log (ts, actor, request, sql, decision, row_count, error) "
            "VALUES (?,?,?,?,?,?,?)",
            (datetime.now(timezone.utc).isoformat(timespec="seconds"), "agent",
             request, sql, decision, row_count, error))
        con.commit()
        con.close()

    def run(self, sql: str, purpose: str = "") -> str:
        ok, reason = is_read_only(sql)
        if not ok:
            self._audit(purpose, sql, "denied", None, reason)
            return (f"query_sql DENIED: {reason}. This tool is read-only; write access "
                    f"is not available. (Logged to the audit trail.)")
        try:
            con = sqlite3.connect(self.db_path)
            cur = con.execute(sql)
            rows = cur.fetchmany(_MAX_ROWS)
            cols = [d[0] for d in cur.description] if cur.description else []
            con.close()
        except Exception as e:  # noqa: BLE001
            self._audit(purpose, sql, "error", None, str(e))
            return f"query_sql error: {e}"
        self._audit(purpose, sql, "allowed", len(rows), None)
        if not rows:
            return "query_sql: (0 rows)"
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        more = "" if len(rows) < _MAX_ROWS else f"\n… (capped at {_MAX_ROWS} rows)"
        return f"{header}\n{'-' * len(header)}\n{body}{more}"


In [5]:
"""src/agents/tools/search_web.py — an external-API tool (web search).

Wraps a web-search API (Tavily) so the agent can reach information beyond the
knowledge base — current events, prices, anything not in the corpus. This is the
"API-calling agent" pattern: the LLM decides it needs fresh external info and
calls out.

Web search has real cost and latency, so the description tells the model to use
it only when the answer isn't in the corpus. For offline runs and tests, the tool
falls back to a local fixture file, so the whole thing works with no network and
no TAVILY_API_KEY.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from pydantic import BaseModel, Field

# Tool defined above

FIXTURES_PATH = Path("data/web_fixtures.json")


class SearchWebArgs(BaseModel):
    query: str = Field(..., description="A concise web-search query.")
    max_results: int = Field(3, ge=1, le=5, description="How many results to return.")


class SearchWebTool(Tool):
    name = "search_web"
    description = (
        "Search the public web and return short result snippets. Use this ONLY when the "
        "answer is not in the internal knowledge base — for current events, live figures, "
        "recent releases, or anything outside company documents. It has cost and latency, "
        "so prefer the internal tools when they can answer. Returns titles and snippets, "
        "not full pages.")
    Args = SearchWebArgs
    idempotent = True            # reading search results doesn't change state
    has_side_effects = False

    def __init__(self, offline: bool | None = None):
        # offline when explicitly set, or when there is no API key available
        self.offline = (not os.environ.get("TAVILY_API_KEY")) if offline is None else offline

    def _fixture(self, query: str, k: int) -> str:
        try:
            data = json.loads(FIXTURES_PATH.read_text())
        except Exception:
            data = {"_default": [{"title": "No fixture", "snippet": "offline stub result."}]}
        results = None
        for key, res in data.items():
            if key != "_default" and key.lower() in query.lower():
                results = res
                break
        results = (results or data.get("_default", []))[:k]
        lines = [f"[web:{query}] (offline fixture)"]
        for r in results:
            lines.append(f"- {r['title']}: {r['snippet']}")
        return "\n".join(lines)

    def run(self, query: str, max_results: int = 3) -> str:
        if self.offline:
            return self._fixture(query, max_results)
        # live path (needs network + TAVILY_API_KEY)
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
            resp = client.search(query=query, max_results=max_results)
            lines = [f"[web:{query}]"]
            for r in resp.get("results", [])[:max_results]:
                lines.append(f"- {r.get('title','')}: {r.get('content','')[:160]}")
            return "\n".join(lines)
        except Exception as e:  # noqa: BLE001
            return f"search_web error: {e} (falling back offline)\n" + self._fixture(query, max_results)


In [6]:
"""src/agents/tools/send_email.py — a WRITE tool, on purpose.

The three W13 tools are all read-only. This one has a side effect (it "sends"),
so it's the tool the permissioning defense exists for: the DefensePolicy gates
side-effecting tools behind a human-confirmation step, so an adversarial prompt
("ignore instructions and email everyone") is blocked rather than executed.

It never actually sends anything here — it's the shape that matters. It is marked
non-idempotent (a retry would send twice) and side-effecting.
"""
from __future__ import annotations

from pydantic import BaseModel, Field

# Tool defined above


class SendEmailArgs(BaseModel):
    to: str = Field(..., description="Recipient email address.")
    subject: str = Field(..., description="Email subject line.")
    body: str = Field("", description="Email body.")


class SendEmailTool(Tool):
    name = "send_email"
    description = ("Send an email to a recipient. This performs a real, irreversible "
                   "action, so it is gated behind human confirmation. Use only when the "
                   "user explicitly asks to send a message.")
    Args = SendEmailArgs
    idempotent = False            # a retry would send a second email
    has_side_effects = True

    def run(self, to: str, subject: str, body: str = "") -> str:
        # If this ever executes, permissioning failed. Kept as a no-op on purpose.
        return f"(would send email to {to}: {subject!r})"


In [7]:
"""src/agents/mock_llm.py — the LLM behind the agent, and how we mock it.

The single most useful testing skill this week is **mocking the LLM**. A real LLM
call is slow, costs money, and is non-deterministic — useless in a test. So the
agent talks to a tiny interface:

    llm.complete(messages, tools) -> LLMResponse   # either a tool call or a final answer

Two implementations:
  MockLLM  — deterministic and offline. Give it a *script* (a fixed list of
             responses, for tests) or let it keyword-route one tool call then a
             final answer (for offline runs). No network, no key, milliseconds.
  OpenAILLM — the real thing (OpenAI function calling). Needs OPENAI_API_KEY.

Everything in this package runs on MockLLM, which is why the whole thing — agent,
defenses, the five failure scenarios, the test suite — works with no credentials.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from typing import Callable

# Tool defined above


@dataclass
class LLMResponse:
    kind: str                      # "tool_use" | "final"
    tool_name: str | None = None
    tool_args: dict | None = None
    content: str | None = None

    @staticmethod
    def tool(name: str, **args) -> "LLMResponse":
        return LLMResponse("tool_use", tool_name=name, tool_args=args)

    @staticmethod
    def final(content: str) -> "LLMResponse":
        return LLMResponse("final", content=content)


class MockLLM:
    """A deterministic stand-in for a function-calling LLM.

    - Scripted:  MockLLM(script=[LLMResponse.tool('calculate', expression='2+2'),
                                 LLMResponse.final('It is 4.')])
                 returns those in order (extra calls repeat the last one).
    - Keyword:   MockLLM()  → routes ONE tool call by keyword, then a final answer
                 synthesised from the observation. Good for offline demos.
    """

    def __init__(self, script: list[LLMResponse] | None = None,
                 router: Callable[[list[dict], list[Tool]], LLMResponse] | None = None):
        self.script = list(script) if script else None
        self.router = router
        self.calls = 0

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        self.calls += 1
        if self.script is not None:
            return self.script.pop(0) if self.script else LLMResponse.final(
                "(no more scripted responses)")
        if self.router is not None:
            return self.router(messages, tools)
        return self._keyword_route(messages, tools)

    # ── default keyword routing: one tool call, then a final answer ──────────
    def _keyword_route(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        # If a tool has already produced an observation, answer from it.
        obs = _last_observation(messages)
        if obs is not None:
            return LLMResponse.final(f"Based on the tool result: {obs}")
        q = _last_user(messages).lower()
        names = {t.name for t in tools}
        if "calculate" in names and re.search(r"\d\s*[-+*/%]\s*\d|percent|%|how much is", q):
            return LLMResponse.tool("calculate", expression=_extract_expr(q))
        sql_kw = ("how many doc", "which department", "what version", "list the", "count",
                  "how many hr", "how many it", "how many finance", "updated", "chunk")
        if "query_sql" in names and any(k in q for k in sql_kw):
            return LLMResponse.tool("query_sql",
                                    sql="SELECT COUNT(*) AS total_documents FROM document_metadata",
                                    purpose="metadata question")
        web_kw = ("latest", "current", "today", "news", "price", "recent", "2026", "who won")
        if "search_web" in names and any(k in q for k in web_kw):
            return LLMResponse.tool("search_web", query=_last_user(messages), max_results=3)
        return LLMResponse.final("I can answer that directly: (no tool needed).")


# ─── real OpenAI adapter (needs a key + network) ────────────────────────────
class OpenAILLM:
    def __init__(self, model: str = "gpt-5-mini"):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model

    def complete(self, messages: list[dict], tools: list[Tool]) -> LLMResponse:
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            tools=[t.openai_schema() for t in tools], tool_choice="auto")
        msg = resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            return LLMResponse.tool(call.function.name, **json.loads(call.function.arguments or "{}"))
        return LLMResponse.final(msg.content or "")


# ─── tiny helpers ───────────────────────────────────────────────────────────
def _last_user(messages: list[dict]) -> str:
    for m in reversed(messages):
        if m.get("role") == "user":
            return str(m.get("content", ""))
    return ""


def _last_observation(messages: list[dict]) -> str | None:
    for m in reversed(messages):
        if m.get("role") == "tool":
            return str(m.get("content", ""))
    return None


def _extract_expr(q: str) -> str:
    m = re.search(r"[0-9][0-9\.\s\+\-\*/%\(\)]*[0-9\)]", q)
    return m.group(0).strip() if m else "0"


### Build the side-database

In [8]:
#!/usr/bin/env python3
"""scripts/seed_db.py — (re)create the capstone side database.

W13 gives the assistant structured data to query as a tool. This builds a small
SQLite database with two tables:

  document_metadata  — one row per document in the knowledge base (the structured
                       side-table the SQL tool queries)
  audit_log          — every SQL query the agent runs is logged here (the audit
                       rail for SQL-as-a-tool)

SQLite is local and offline, so the whole SQL-tool exercise runs with no network
and no credentials. Run this once (smoke_test.sh does it for you):

    python3 scripts/seed_db.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

DB_PATH = Path(".") / "data" / "capstone_side.db"

# A small, realistic slice of the Enterprise Knowledge Assistant's corpus,
# across the three departments the capstone covers.
DOCS = [
    # source, title, doc_type, department, version, updated_at, chunk_count
    ("hr-001", "Leave & Time-Off Policy",          "policy",   "HR",      "v3", "2026-01-14", 42),
    ("hr-002", "Code of Conduct",                  "policy",   "HR",      "v2", "2025-11-02", 55),
    ("hr-003", "Remote Work Guidelines",           "guideline","HR",      "v4", "2026-02-20", 28),
    ("hr-004", "Onboarding Handbook",              "handbook", "HR",      "v1", "2025-06-30", 90),
    ("it-001", "Acceptable Use Policy",            "policy",   "IT",      "v5", "2026-01-30", 37),
    ("it-002", "Security Incident Runbook",        "runbook",  "IT",      "v2", "2026-02-11", 33),
    ("it-003", "VPN & Access Setup",               "guideline","IT",      "v3", "2025-12-05", 19),
    ("it-004", "Data Classification Standard",     "standard", "IT",      "v2", "2026-01-08", 24),
    ("fin-001","Expense Reimbursement Policy",     "policy",   "Finance", "v3", "2026-01-22", 31),
    ("fin-002","Procurement Guidelines",           "guideline","Finance", "v1", "2025-09-15", 40),
    ("fin-003","Travel Policy",                    "policy",   "Finance", "v2", "2025-10-19", 26),
    ("fin-004","Corporate Card Terms",             "terms",    "Finance", "v4", "2026-02-01", 18),
]


def seed(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE document_metadata (
            id           INTEGER PRIMARY KEY,
            source       TEXT NOT NULL,
            title        TEXT NOT NULL,
            doc_type     TEXT NOT NULL,
            department   TEXT NOT NULL,
            version      TEXT NOT NULL,
            updated_at   TEXT NOT NULL,
            chunk_count  INTEGER NOT NULL
        )""")
    cur.executemany(
        "INSERT INTO document_metadata "
        "(source,title,doc_type,department,version,updated_at,chunk_count) "
        "VALUES (?,?,?,?,?,?,?)", DOCS)
    cur.execute("""
        CREATE TABLE audit_log (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            ts         TEXT NOT NULL,
            actor      TEXT NOT NULL,
            request    TEXT,
            sql        TEXT,
            decision   TEXT NOT NULL,
            row_count  INTEGER,
            error      TEXT
        )""")
    con.commit()
    con.close()
    return db_path

seed()
print('side-DB built:', DB_PATH)


side-DB built: data/capstone_side.db


## 1 · The ported agent graph

The W14 loop as a LangGraph — think (agentic) ↔ tool (deterministic), an accumulating step log, and an iteration guard. Behaviourally equivalent to the raw agent; different shape.

In [9]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

TOOLS = {t.name: t for t in [SearchWebTool(offline=True), QuerySqlTool(), CalculateTool(), SendEmailTool()]}
llm = MockLLM()

class AgentState(TypedDict):
    question: str
    steps: Annotated[list, lambda a, b: a + b]
    answer: str
    current_tool: dict | None
    iters: int
    cost: float
    flagged: str

COST_PER_CALL = 0.03   # a simple deterministic cost model (per LLM node call)

def think(s):
    msgs = [{'role': 'system', 'content': 'agent'}, {'role': 'user', 'content': s['question']}]
    for st in s['steps']:
        msgs.append({'role': 'tool', 'content': st['obs']})
    r = llm.complete(msgs, list(TOOLS.values()))
    upd = {'iters': s['iters'] + 1, 'cost': s['cost'] + COST_PER_CALL}
    if r.kind == 'final':
        upd['answer'] = r.content; upd['current_tool'] = None
    else:
        upd['current_tool'] = {'name': r.tool_name, 'args': r.tool_args or {}}
    return upd

def tool(s):
    t = s['current_tool']; obs = TOOLS[t['name']].call(t['args'])
    return {'steps': [{'tool': t['name'], 'obs': obs}], 'current_tool': None}

def route_after_think(s):
    if s.get('answer') or s['iters'] >= 5:  return 'final'
    return 'tool' if s.get('current_tool') else 'final'

## 2 · Guardrail node 1 — PII check (deterministic)

A deterministic node that runs **before any LLM call**. In production this is **Presidio** (from W8); offline we use a lightweight regex detector — the *point* is that a deterministic node enforces the policy **every time**, not an LLM that might slip. If PII is found, it redacts and flags.

Keep two things separate, though: the determinism is the **enforcement** (redact on every match — testable, auditable). It is *not* a guarantee of **detection** — a regex (and even Presidio) has limited recall, so what counts as PII and whether you catch all of it is a separate, imperfect problem. Measure the detector's recall on its own.

In [10]:
import re
PII = re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+'          # email
                 r'|\b\d{3}[-.]?\d{2}[-.]?\d{4}\b'   # SSN-like
                 r'|\b\d{10}\b')                       # 10-digit phone

def pii_check(s):
    if PII.search(s['question']):
        return {'question': PII.sub('[REDACTED]', s['question']), 'flagged': 'pii-redacted'}
    return {'flagged': ''}

# quick unit check of the node in isolation:
print(pii_check({'question': 'email me at bob@acme.com'}))
print(pii_check({'question': 'what is the leave policy?'}))

{'question': 'email me at [REDACTED]', 'flagged': 'pii-redacted'}
{'flagged': ''}


## 3 · Guardrail node 2 — cost limit (deterministic)

A deterministic node that checks accumulated **cost** in state; over the threshold, it routes to a **graceful-degradation** final instead of another expensive LLM call. Threshold ≈ 2× the average call cost (the W11 snapshot heuristic) — rarely triggers on normal use, but catches runaway loops.

In [11]:
COST_LIMIT = 0.10

def degrade(s):
    return {'answer': '(cost limit reached — returning a partial answer.)'}

def route_after_tool(s):
    return 'degrade' if s['cost'] > COST_LIMIT else 'think'   # deterministic cost gate

print('under limit →', route_after_tool({'cost': 0.06}))
print('over limit  →', route_after_tool({'cost': 0.15}))

under limit → think
over limit  → degrade


## 4 · The guardrailed agent graph

Wire it all together: **`pii_check` → think ↔ tool** (with the cost gate on the tool→think edge) → **`degrade`** or `END`. The agentic core (`think`) is bracketed by deterministic guardrails — the W16 shape.

In [12]:
g = StateGraph(AgentState)
g.add_node('pii_check', pii_check)
g.add_node('think', think); g.add_node('tool', tool); g.add_node('degrade', degrade)
g.add_edge(START, 'pii_check')
g.add_edge('pii_check', 'think')
g.add_conditional_edges('think', route_after_think, {'tool': 'tool', 'final': END})
g.add_conditional_edges('tool', route_after_tool, {'think': 'think', 'degrade': 'degrade'})
g.add_edge('degrade', END)
guarded = g.compile()
print(guarded.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	pii_check(pii_check)
	think(think)
	tool(tool)
	degrade(degrade)
	__end__([<p>__end__</p>]):::last
	__start__ --> pii_check;
	pii_check --> think;
	think -. &nbsp;final&nbsp; .-> __end__;
	think -.-> tool;
	tool -.-> degrade;
	tool -.-> think;
	degrade --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [13]:
def run(q):
    out = guarded.invoke({'question': q, 'steps': [], 'answer': '', 'current_tool': None,
                          'iters': 0, 'cost': 0.0, 'flagged': ''})
    print(f'Q: {q}')
    print(f"   flagged: {out['flagged'] or '—'} | tools: {[s['tool'] for s in out['steps']]} | "
          f"cost: ${out['cost']:.2f} | answer: {out['answer'][:55]}")

run('How many documents are there?')      # normal path
run('email me the policy at bob@acme.com')  # PII guardrail redacts the input

Q: How many documents are there?
   flagged: — | tools: ['query_sql'] | cost: $0.06 | answer: Based on the tool result: total_documents
-------------
Q: email me the policy at bob@acme.com
   flagged: pii-redacted | tools: [] | cost: $0.03 | answer: I can answer that directly: (no tool needed).


## 5 · Resilience primitives — `RetryPolicy`, fallback, `interrupt`

LangGraph **names** the W14 defenses as graph elements — easier to audit and visualize. They don't replace W14's logic; they express it as graph structure.

In [14]:
from langgraph.types import RetryPolicy

# RETRY: wrap a flaky node with a retry policy (maps to W14 retry + backoff)
attempts = {'n': 0}
class RS(TypedDict):
    v: int
def flaky(s):
    attempts['n'] += 1
    if attempts['n'] < 3: raise RuntimeError('transient 429')
    return {'v': s['v'] + 1}
g = StateGraph(RS)
g.add_node('flaky', flaky, retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.0, retry_on=RuntimeError))
g.add_edge(START, 'flaky'); g.add_edge('flaky', END)
print('RetryPolicy → succeeded after', end=' '); print(attempts, '→', g.compile().invoke({'v': 10}))

RetryPolicy → succeeded after {'n': 3} → {'v': 11}


**Fallback** is just an edge from a failing node to a degradation node (we used exactly that with the cost gate → `degrade`). **HITL** is `interrupt()` — a node that *pauses* the graph for human input, then resumes. It's powerful but conceptually heavy, so this is a **preview** — W17 is the deep dive:

In [15]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

class Approval(TypedDict):
    action: str; approved: str
def ask_human(s):
    decision = interrupt({'action': s['action']})   # pause here for a human
    return {'approved': decision}
g = StateGraph(Approval); g.add_node('ask_human', ask_human)
g.add_edge(START, 'ask_human'); g.add_edge('ask_human', END)
app = g.compile(checkpointer=MemorySaver())   # interrupts need a checkpointer
cfg = {'configurable': {'thread_id': 'req1'}}
paused = app.invoke({'action': 'send_email', 'approved': ''}, cfg)
print('paused, waiting for human:', paused.get('__interrupt__', 'interrupted'))
resumed = app.invoke(Command(resume='approved'), cfg)   # human says yes → resume
print('resumed →', resumed['approved'])

paused, waiting for human: [Interrupt(value={'action': 'send_email'}, id='585c0b1e16800c7f72c45165fe052ad8')]
resumed → approved


## 6 · Parallel fan-out / fan-in with `Send`

When a question benefits from several tools **at once** (temperature *and* price *and* inventory), fan out to parallel workers and fan in to a synthesizer. `Send('node', payload)` dispatches a dynamic number of parallel branches; a reducer collects their results.

In [16]:
from langgraph.types import Send
import operator

class FanState(TypedDict):
    question: str
    plan: list                                  # which tools to run in parallel
    results: Annotated[list, operator.add]      # reducer collects parallel outputs
    answer: str

def planner(s):  return {'plan': [('calculate', {'expression': '20*3'}),
                                  ('query_sql', {'sql': 'SELECT COUNT(*) FROM document_metadata', 'purpose': 'count'}),
                                  ('search_web', {'query': 'latest AI agents 2026', 'max_results': 1})]}
def fan_out(s): return [Send('worker', {'call': call}) for call in s['plan']]   # parallel dispatch
def worker(s):
    name, args = s['call']; return {'results': [(name, TOOLS[name].call(args)[:40])]}
def synthesize(s): return {'answer': f"combined {len(s['results'])} parallel results"}

g = StateGraph(FanState)
g.add_node('planner', planner); g.add_node('worker', worker); g.add_node('synthesize', synthesize)
g.add_edge(START, 'planner')
g.add_conditional_edges('planner', fan_out, ['worker'])
g.add_edge('worker', 'synthesize'); g.add_edge('synthesize', END)
out = g.compile().invoke({'question': 'x', 'plan': [], 'results': [], 'answer': ''})
print('ran', len(out['results']), 'tools in parallel:')
for name, r in out['results']: print(f'   {name}: {r}')
print('→', out['answer'])

ran 3 tools in parallel:
   calculate: 20*3 = 60
   query_sql: COUNT(*)
--------
12
   search_web: [web:latest AI agents 2026] (offline fix
→ combined 3 parallel results


## 7 · Previews — persistence & streaming (deeper in W17)

Two LangGraph components you'll use later, shown briefly. **Persistence** (`MemorySaver` + a `thread_id`) makes graph state survive *across* invocations — a natural home for the W15 memory layer; the deep dive is **W17** (where HITL's interrupt/resume uses the checkpointer).

In [17]:
from langgraph.checkpoint.memory import MemorySaver
class Conv(TypedDict):
    turns: Annotated[list, operator.add]
g = StateGraph(Conv); g.add_node('turn', lambda s: {'turns': ['t']})
g.add_edge(START, 'turn'); g.add_edge('turn', END)
app = g.compile(checkpointer=MemorySaver())
cfg = {'configurable': {'thread_id': 'user-42'}}
app.invoke({'turns': []}, cfg); app.invoke({'turns': []}, cfg)   # two separate calls
print('state persisted across calls:', len(app.get_state(cfg).values['turns']), 'turns')

state persisted across calls: 2 turns


**Streaming** (`.stream()`) emits progress as the graph runs — one update per node — which powers responsive UX; the deep dive is **W17**:

In [18]:
for update in app.stream({'turns': []}, {'configurable': {'thread_id': 'user-99'}}, stream_mode='updates'):
    print('  step update:', update)

  step update: {'turn': {'turns': ['t']}}


## 8 · Behavioural equivalence & wrap

The LangGraph agent should score the **same** on your golden set as the raw W14 agent — you changed the *shape*, not the behaviour. That equivalence, plus the `draw_mermaid` diagram and the two deterministic guardrails, is defensible architecture for **DR #3** (W18).

You've now: ported the agent to a `StateGraph`; bracketed the agentic core with deterministic **PII** and **cost** guardrails; expressed retry/fallback/HITL as graph elements; run tools in **parallel** with `Send`; and previewed persistence + streaming. Next: **W17 — Planning, Streaming & Human-in-the-Loop** (the 'danger zone': HITL becomes a recurring thread).

*These notebooks are for seeing and explaining the graph; in your capstone you lift this into `src/agents/graph.py`, commit the `.draw_mermaid()` output to `docs/`, and add the two guardrail tests.*